# 1. Install + Import Requirements

In [ ]:
%pip install stable-baselines3[extra]

import os

# Change this path to the specific folder in your Drive where the .obj files are locate
os.chdir('/app')
print(f"Current working directory: {os.getcwd()}")

# List files to verify existence
print("Files in directory:", os.listdir('.'))

os.environ['MUJOCO_GL'] = 'egl'

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

%pip install mujoco mediapy numpy

import mujoco
import mediapy as media
import numpy as np

%pip install ultralytics

from ultralytics import YOLO

%pip install opencv-python

# 2. Define MuJoCo XML Scene

In [ ]:
xml_string = """
<mujoco>
  <compiler angle="degree"/>
  
  <asset>
    <!-- Define the .obj files as mesh assets -->
    <mesh name="mug_mesh" file="mug.obj"/>
    <mesh name="can_opener_mesh" file="can_opener.obj"/>
    <mesh name="action_fig_mesh" file="action_fig.obj"/>
    <mesh name="shoe_mesh" file="shoe.obj"/>
  </asset>

  <worldbody>
    <light pos="0 0 1.5" dir="0 0 -1" directional="true" diffuse="0.8 0.8 0.8"/>
    <geom type="plane" size="1 1 0.1" rgba="0.9 0.9 0.9 1"/>
    
    <!-- Object: Mug -->
    <body name="mug" pos="0 0 0.1">
      <freejoint/>
      <geom type="mesh" mesh="mug_mesh" rgba="0.8 0.2 0.2 1"/>
    </body>
    
    <!-- Object: Can Opener -->
    <body name="can_opener" pos="0.1 0 0.1">
      <freejoint/>
      <geom type="mesh" mesh="can_opener_mesh" rgba="0.2 0.8 0.2 1"/>
    </body>
    
    <!-- Object: Action Figure -->
    <body name="action_fig" pos="-0.1 0 0.1">
      <freejoint/>
      <geom type="mesh" mesh="action_fig_mesh" rgba="0.2 0.2 0.8 1"/>
    </body>
    
    <!-- Object: Shoe -->
    <body name="shoe" pos="0 0.1 0.1">
      <freejoint/>
      <geom type="mesh" mesh="shoe_mesh" rgba="0.5 0.5 0.5 1"/>
    </body>
  </worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(xml_string)
data = mujoco.MjData(model)

# 3. Generate Training Data for YOLO Finetuning
This will be labelled bounding boxes from openCV of the objects initialized to random transforms

In [ ]:
import cv2
import yaml

skip_perception_training = os.path.exists("runs/obb/current")

dataset_dir = "datasets/mujoco_obb"

# all objects have class 0, because we dont care what they are - just need to sweep them.
classes = {"mug": 0, "can_opener": 0, "action_fig": 0, "shoe": 0}
object_names = list(classes.keys())         # must be after classes dict

# use MuJoCo's built in segmenter
seg_renderer = mujoco.Renderer(model, 480, 640)
seg_renderer.enable_segmentation_rendering()
renderer = mujoco.Renderer(model, 480, 640)  # must be after seg_renderer

if skip_perception_training:
    print("Skipping dataset generation (using runs/obb/current).")
else:

    # set up datasets
    os.makedirs(f"{dataset_dir}/images/train", exist_ok=True)
    os.makedirs(f"{dataset_dir}/labels/train", exist_ok=True)

    # generate 100 training images
    print("Generating synthetic data. This will take a moment...")
    frames_to_generate = 100
    vis_img = None

    for i in range(frames_to_generate):
        # random positions and rotations for each frame
        for obj_name in object_names:
            body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, obj_name)
            
            rand_x, rand_y = np.random.uniform(-0.4, 0.4, size=2)

            # add random rotation
            rand_angle = np.random.uniform(0, 2 * np.pi)
            q_z = [np.cos(rand_angle/2), 0, 0, np.sin(rand_angle/2)]
            
            qpos_start = model.jnt_qposadr[model.body_jntadr[body_id]]
            data.qpos[qpos_start:qpos_start+3] = [rand_x, rand_y, 0.2]
            data.qpos[qpos_start+3:qpos_start+7] = q_z
            
        mujoco.mj_forward(model, data)
        num_steps = np.random.randint(10, 100)  # let the objects fall a random amount, would like mid-air support
        for _ in range(num_steps):  # roll out physics
            mujoco.mj_step(model, data)
            
        # render scene and segments
        renderer.update_scene(data)
        rgb_img = renderer.render()
        
        seg_renderer.update_scene(data)
        seg_img = seg_renderer.render()
        
        img_h, img_w = rgb_img.shape[:2]
        
        # save the last frame to see if the model is working correctly
        if i == frames_to_generate - 1:
            vis_img = rgb_img.copy()

        # now we can make the label for this data point
        label_file = open(f"{dataset_dir}/labels/train/frame_{i}.txt", "w")
        
        for obj_name, class_idx in classes.items():
            body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, obj_name)
            
            # get the MuJoCo geoms
            geom_start = model.body_geomadr[body_id]
            geom_num = model.body_geomnum[body_id]
            
            # create binary mask for this object
            mask = np.isin(seg_img[:, :, 0], range(geom_start, geom_start + geom_num)).astype(np.uint8) * 255
            
            # use openCV to get the bounding box
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            if contours:
                c = max(contours, key=cv2.contourArea)
                # only save the bounding box if not obscured or offscreen
                if cv2.contourArea(c) > 50: 
                    # minAreaRect gets the box corners
                    rect = cv2.minAreaRect(c)
                    box = cv2.boxPoints(rect)
                    
                    if vis_img is not None:
                        # draw a green box in the visualization image for us to see if its working
                        cv2.drawContours(vis_img, [np.int32(box)], 0, (0, 255, 0), 2)
                    
                    # must normalize coordinates between 0.0 and 1.0 to work in YOLO
                    box[:, 0] /= img_w
                    box[:, 1] /= img_h
                    
                    # yolo format: "class_idx x1 y1 x2 y2 x3 y3 x4 y4"
                    label_str = " ".join([f"{coord:.5f}" for coordinate in box for coord in coordinate])
                    label_file.write(f"{class_idx} {label_str}\n")
                    
        label_file.close()
        
        # create image for this data point, opencv uses BGR format
        cv2.imwrite(f"{dataset_dir}/images/train/frame_{i}.jpg", cv2.cvtColor(rgb_img, cv2.COLOR_RGB2BGR))

    print("Synthetic Dataset Generation Complete!")

    # view the final frame with the visualization as a sanity check
    if vis_img is not None:
        print("Visual sanity check:")
        media.show_image(vis_img)

# 4. Fine Tune YOLO
Use the data set of labelled images we just generated.

In [ ]:
if skip_perception_training:
    print("Skipping YOLO training (using runs/obb/current).")
else:
    # ultralytics needs dataset.yaml, so make it
    yaml_config = {
        'path': os.path.abspath(dataset_dir),
        'train': 'images/train',
        'val': 'images/train', # WILL CHANGE: need to use actual validation set
        # only one class
        'names': {0: "object"}
    }

    with open('mujoco_dataset.yaml', 'w') as f:
        yaml.dump(yaml_config, f)

    # import base YOLO
    model_obb = YOLO('yolov8n-obb.pt')

    # train
    results = model_obb.train(data='mujoco_dataset.yaml', epochs=25, imgsz=640, device="cpu", verbose=False)

    print("Training finished!")

# 4. Test Perception Module

In [ ]:
class PerceptionModule:
    """
    Combines YOLO 2D Object Detection with Depth maps to estimate
    the 3D transform (position and orientation) of detected objects
    relative to the camera.
    """
    def __init__(self, weights_path, fovy_degrees=45.0):
        self.model = YOLO(weights_path)
        # default camera has 45 degree vertical field of view
        self.fovy_degrees = fovy_degrees

    def _rotation_from_obb(self, rotation_rad):
        # Build a camera-frame rotation from the OBB angle in the image plane.
        c = float(np.cos(rotation_rad))
        s = float(np.sin(rotation_rad))
        x_axis = np.array([c, s, 0.0], dtype=np.float32)
        z_axis = np.array([0.0, 0.0, -1.0], dtype=np.float32)
        y_axis = np.cross(z_axis, x_axis)
        x_axis /= np.linalg.norm(x_axis) + 1e-8
        y_axis /= np.linalg.norm(y_axis) + 1e-8
        z_axis = np.cross(x_axis, y_axis)
        return np.stack([x_axis, y_axis, z_axis], axis=1)

    def get_transforms(self, rgb_image, depth_image):
        # run model
        results = self.model(rgb_image, verbose=False)[0]

        objects = []
        if results.obb is not None:
            h, w = rgb_image.shape[:2]

            # get camera intrinsics
            fovy_rad = np.deg2rad(self.fovy_degrees)
            fy = (h / 2.0) / np.tan(fovy_rad / 2.0)
            fx = fy
            cx = w / 2.0
            cy = h / 2.0

            for box in results.obb:
                # get OBB parameters
                x_center, y_center, width, height, rotation = box.xywhr[0].cpu().numpy()
                class_id = int(box.cls[0].item())
                confidence = float(box.conf[0].item())

                # get depth at each object's center
                u = int(np.clip(x_center, 0, w - 1))
                v = int(np.clip(y_center, 0, h - 1))
                z = depth_image[v, u]

                # use deprojection formula:
                x = (u - cx) * z / fx
                y = (v - cy) * z / fy

                rot_cam = self._rotation_from_obb(rotation)
                tf_cam = np.eye(4, dtype=np.float32)
                tf_cam[:3, :3] = rot_cam
                tf_cam[:3, 3] = np.array([x, y, z], dtype=np.float32)

                transform = {
                    "class": results.names[class_id],
                    "position_camera_frame": np.array([x, y, z], dtype=np.float32),
                    "rotation_rad": float(rotation),
                    "rotation_camera_frame": rot_cam,
                    "transform_camera_frame": tf_cam,
                    "confidence": confidence,
                    "pixel_u": u,
                    "pixel_v": v
                }
                objects.append(transform)

        # return the list of object transforms and the annotated image
        return objects, results.plot()

# 5. Scene with Robot (Broom + Franka Panda)
Build the combined MuJoCo scene using `mujoco.MjSpec`.

The broom mesh is generated **programmatically** — correct dimensions,
zero licensing issues, no external conversion needed.

In [ ]:
import subprocess

# ── 0. Generate broom.obj if not already present ──────────────────────────
def _make_broom_obj(path, n_seg=20):
    import numpy as np
    verts, faces = [], []
    r, hz0, hz1 = 0.015, 0.05, 0.75
    for i in range(n_seg):
        a = 2*np.pi*i/n_seg; verts.append((r*np.cos(a), r*np.sin(a), hz0))
    for i in range(n_seg):
        a = 2*np.pi*i/n_seg; verts.append((r*np.cos(a), r*np.sin(a), hz1))
    verts += [(0,0,hz0),(0,0,hz1)]
    bc, tc = 2*n_seg+1, 2*n_seg+2
    for i in range(n_seg):
        v0,v1,v2,v3 = i+1, i%n_seg+2, n_seg+i%n_seg+2, n_seg+i+1
        faces += [(v0,v1,v2),(v0,v2,v3)]
    for i in range(n_seg): faces.append((bc, i%n_seg+2, i+1))
    for i in range(n_seg): faces.append((tc, n_seg+i+1, n_seg+i%n_seg+2))
    bx,by,bz0,bz1 = 0.12,0.03,0.0,0.05; o = len(verts)+1
    for bv in [(-bx,-by,bz0),(bx,-by,bz0),(bx,by,bz0),(-bx,by,bz0),
               (-bx,-by,bz1),(bx,-by,bz1),(bx,by,bz1),(-bx,by,bz1)]:
        verts.append(bv)
    faces += [(o,o+2,o+1),(o,o+3,o+2),(o+4,o+5,o+6),(o+4,o+6,o+7),
              (o,o+1,o+5),(o,o+5,o+4),(o+1,o+2,o+6),(o+1,o+6,o+5),
              (o+2,o+3,o+7),(o+2,o+7,o+6),(o+3,o,o+4),(o+3,o+4,o+7)]
    with open(path,"w") as f:
        f.write("# Parametric broom: handle r=0.015m z=[0.05,0.75]; head 0.24x0.06x0.05m\n")
        for v in verts: f.write(f"v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n")
        for fc in faces: f.write(f"f {fc[0]} {fc[1]} {fc[2]}\n")
    print(f"Generated {path}  ({len(verts)} verts, {len(faces)} faces)")

if not os.path.exists("broom.obj"):
    _make_broom_obj("broom.obj")
else:
    print("broom.obj already present")

# ── 1. Download Franka Panda model (mujoco_menagerie) ─────────────────────
if not os.path.exists("mujoco_menagerie"):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/google-deepmind/mujoco_menagerie.git"],
        check=True,
    )

PANDA_XML = "mujoco_menagerie/franka_emika_panda/panda.xml"

# ── 2. Base scene XML (broom mesh included directly) ─────────────────────
# Broom geometry (single geom = visual + collision):
#   Handle : cylinder r=0.015 m,  z = [0.05, 0.75]
#   Head   : box ±0.12 x ±0.03 x [0, 0.05] m
# Grasp point for IK: broom_world_pos + [0, 0, 0.55]  (upper handle)
BASE_XML = """
<mujoco model="sweep_scene">
  <compiler angle="radian"/>
  <option gravity="0 0 -9.81"/>
  
  <default>
    <!-- Increase global default friction (sliding, torsional, rolling) -->
    <geom friction="2.0 0.05 0.0001"/>
  </default>

  <asset>
    <mesh name="mug_mesh"        file="mug.obj"/>
    <mesh name="can_opener_mesh" file="can_opener.obj"/>
    <mesh name="action_fig_mesh" file="action_fig.obj"/>
    <mesh name="shoe_mesh"       file="shoe.obj"/>
    <mesh name="broom_mesh"      file="broom.obj"/>
  </asset>
  <worldbody>
    <light pos="0 0 2" dir="0 0 -1" directional="true" diffuse="0.8 0.8 0.8"/>
    <geom type="plane" size="1 1 0.1" rgba="0.9 0.9 0.9 1"/>
    <!-- Fixed overhead camera used by PerceptionModule -->
    <camera name="overhead_cam" pos="0 0 1.5" euler="0 0 0" fovy="45"/>
    <!-- Broom: freejoint so it can be grasped and lifted -->
    <body name="broom" pos="0.3 0.0 0.0">
      <freejoint name="broom_joint"/>
      <!-- Visual geom uses the mesh -->
      <geom name="broom_geom_viz" type="mesh" mesh="broom_mesh"
            rgba="0.65 0.38 0.12 1" contype="0" conaffinity="0"/>
      <!-- Collision geoms using primitives to avoid convex hull artifacts -->
      <!-- Handle: radius 0.015, height extent [0.05, 0.75] -->
      <!-- Center z = (0.75+0.05)/2 = 0.4. Half-length = (0.75-0.05)/2 = 0.35 -->
      <geom name="broom_col_handle" type="cylinder" size="0.015 0.35" pos="0 0 0.4"
            rgba="0 0 0 0" contype="1" conaffinity="2" friction="3.0 0.1 0.001"/>
      <!-- Head: box from x=-0.12..0.12, y=-0.03..0.03, z=0..0.05 -->
      <!-- Center z=0.025. Half-size: 0.12 0.03 0.025 -->
      <geom name="broom_col_head" type="box" size="0.12 0.03 0.025" pos="0 0 0.025"
            rgba="0 0 0 0" contype="1" conaffinity="2"/>
    </body>
    <!-- Sweepable objects -->
    <body name="mug"        pos="0    0    0.1"><freejoint/><geom type="mesh" mesh="mug_mesh"        rgba="0.8 0.2 0.2 1"/></body>
    <body name="can_opener" pos="0.1  0    0.1"><freejoint/><geom type="mesh" mesh="can_opener_mesh" rgba="0.2 0.8 0.2 1"/></body>
    <body name="action_fig" pos="-0.1 0    0.1"><freejoint/><geom type="mesh" mesh="action_fig_mesh" rgba="0.2 0.2 0.8 1"/></body>
    <body name="shoe"       pos="0    0.1  0.1"><freejoint/><geom type="mesh" mesh="shoe_mesh"       rgba="0.5 0.5 0.5 1"/></body>
  </worldbody>
</mujoco>
"""

# ── 3. Compose with MjSpec (Panda only — broom is already in BASE_XML) ────
scene_spec = mujoco.MjSpec.from_string(BASE_XML)
panda_spec = mujoco.MjSpec.from_file(PANDA_XML)

# We want torque control across the arm for RL, so let's modify the arm's actuators 
# from position/PD controllers ("general" with biasprm) to pure "motor" elements.
# The gripper (actuator 7) remains in position control so we just adjust the first 7.
for i in range(7):
    act = panda_spec.actuators[i]
    act.set_to_motor()
    act.biastype = mujoco.mjtBias.mjBIAS_NONE
    act.gaintype = mujoco.mjtGain.mjGAIN_FIXED
    act.gainprm = [1.0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    act.biasprm = [0.0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    
    # Apply standard Panda torque limits
    act.forcerange = [-87.0, 87.0] if i < 4 else [-12.0, 12.0]

# Increase the physical joint limits of the gripper for simulation (default max is 0.04 per finger)
for j in panda_spec.joints:
    if "finger_joint" in j.name:
        j.range = [0.0, 0.08] # double the opening distance to 8cm per finger (16cm total)

# The generic actuator8 handles the fingers via a split tendon. It is bound to 0-255 ctrlrange.
# We also need to double its implicit position target scaling.
# Original gain was 0.015686... meaning `target_pos = ctrl * 0.04 / 255`
panda_spec.actuators[-1].gainprm[0] = 0.08 / 255.0

# Add EE site to panda hand (menagerie panda.xml has no sites)
panda_hand = next(b for b in panda_spec.bodies if b.name == "hand")

# Disable the palm's single convex hull to allow deep grasping between fingers
for geom in panda_hand.geoms:
    if getattr(getattr(geom, "mesh", None), "name", "") == "hand_c":
        geom.contype = 0
        geom.conaffinity = 0

# Add large gripper plates to the fingers to increase grasp surface area
def find_body_in_spec(spec, name):
    for b in spec.bodies:
        if b.name == name: return b
    return None

for finger_name in ["left_finger", "right_finger"]:
    finger_body = find_body_in_spec(panda_spec, finger_name)
    if finger_body:
        # Check if we didn't already add one by accident
        plate = finger_body.add_geom()
        plate.type = mujoco.mjtGeom.mjGEOM_BOX
        # Local +Y is AWAY from the gap! The gap is in the -Y direction.
        # X is the vertical height of the fingers, Z is the forward reach.
        plate.size = [0.02, 0.004, 0.025]
        plate.pos = [0.0, 0.0, 0.045] # Shifted into the -Y gap
        plate.rgba = [0.2, 0.8, 0.2, 1] # Green plate to easily see it
        plate.friction = [3.0, 0.1, 0.001]
        plate.contype = 2
        plate.conaffinity = 1

        # Add "front" ridge (to stop broom sliding out the front)
        ridge1 = finger_body.add_geom()
        ridge1.type = mujoco.mjtGeom.mjGEOM_BOX
        ridge1.size = [0.02, 0.005, 0.004]  # Tall in X, thick in Y, narrow in Z
        ridge1.pos = [0.0, -0.009, 0.045 + 0.025]   # Front end of the plate (Z) and deep in gap (+Y)
        ridge1.rgba = [0.8, 0.2, 0.2, 1]     # Red to distinguish them
        ridge1.friction = [3.0, 0.1, 0.001]
        ridge1.contype = 2
        ridge1.conaffinity = 1

        # Add "back" ridge (to stop broom sliding back into the hand)
        ridge2 = finger_body.add_geom()
        ridge2.type = mujoco.mjtGeom.mjGEOM_BOX
        ridge2.size = [0.02, 0.005, 0.004]
        ridge2.pos = [0.0, -0.009, 0.045 - 0.025]   # Back end of the plate (Z)
        ridge2.rgba = [0.8, 0.2, 0.2, 1]
        ridge2.friction = [3.0, 0.1, 0.001]
        ridge2.contype = 2
        ridge2.conaffinity = 1

        # Stiffen contacts so the ridges resist interpenetration when the gripper closes.
        for geom in (plate, ridge1, ridge2):
            geom.margin = 0.002
            geom.gap = 0.0005
            geom.solref = [0.005, 1.0]
            geom.solimp = [0.9, 0.95, 0.001, 0.5, 2.0]

ee_site      = panda_hand.add_site()
ee_site.name = "attachment_site"
ee_site.pos  = [0.0, 0.0, 0.1]   # ~10 cm toward fingertips in hand local frame

panda_frame       = scene_spec.worldbody.add_frame()
panda_frame.pos   = [0.7, 0.0, 0.0]
panda_frame.quat  = [0.0, 0.0, 0.0, 1.0] # face the table (π around z)
scene_spec.attach(panda_spec, prefix="panda_", frame=panda_frame)

# ── 4. Compile ─────────────────────────────────────────────────────────────
robot_model = scene_spec.compile()
robot_data  = mujoco.MjData(robot_model)
mujoco.mj_forward(robot_model, robot_data)

# ── 5. Resolve names ──────────────────────────────────────────────────────
site_names = [mujoco.mj_id2name(robot_model, mujoco.mjtObj.mjOBJ_SITE, i)
              for i in range(robot_model.nsite)]
print("Sites in scene:", site_names)

PANDA_JOINTS  = [f"panda_joint{i}" for i in range(1, 8)]
FINGER_JOINTS = ["panda_finger_joint1", "panda_finger_joint2"]
# Confirm EE_SITE matches one of the printed site names above
EE_SITE = "panda_attachment_site"

print(f"robot_model compiled — nq: {robot_model.nq}, nv: {robot_model.nv}")

# 6. Retrain YOLO with Broom Class
Add the broom (class 1) to synthetic training data and retrain YOLOv8-OBB.

In [ ]:
if skip_perception_training:
    print("Skipping dataset generation and training for broom (using runs/obb/current).")
    BROOM_WEIGHTS = "runs/obb/current/weights/best.pt"
    print(f"BROOM_WEIGHTS: {BROOM_WEIGHTS}")
else:
    DATASET_DIR_R = "datasets/mujoco_obb_with_broom"
    os.makedirs(f"{DATASET_DIR_R}/images/train", exist_ok=True)
    os.makedirs(f"{DATASET_DIR_R}/labels/train", exist_ok=True)

    # Sweepable objects -> class 0, broom -> class 1
    classes_r = {
        "mug": 0, "can_opener": 0, "action_fig": 0, "shoe": 0,
        "broom": 1,
    }

    seg_renderer_r = mujoco.Renderer(robot_model, 480, 640)
    seg_renderer_r.enable_segmentation_rendering()
    renderer_r = mujoco.Renderer(robot_model, 480, 640)

    object_names_r = list(classes_r.keys())

    print("Generating synthetic data with broom. This will take a moment...")
    frames_to_generate = 100
    vis_img_r = None

    for i in range(frames_to_generate):
        for obj_name in object_names_r:
            body_id = mujoco.mj_name2id(robot_model, mujoco.mjtObj.mjOBJ_BODY, obj_name)
            rand_x, rand_y = np.random.uniform(-0.35, 0.35, size=2)

            # 80% chance to be relatively upright (z-axis rotation) and 20% fully random
            if np.random.rand() > 0.2:
                rand_yaw = np.random.uniform(0, 2 * np.pi)
                q_rand = np.array([np.cos(rand_yaw/2), 0.0, 0.0, np.sin(rand_yaw/2)])
            else:
                q_rand = np.random.randn(4)
                q_rand /= np.linalg.norm(q_rand)

            qpos_start = robot_model.jnt_qposadr[robot_model.body_jntadr[body_id]]
            robot_data.qpos[qpos_start:qpos_start + 3] = [rand_x, rand_y, 0.15]
            robot_data.qpos[qpos_start + 3:qpos_start + 7] = q_rand

        mujoco.mj_forward(robot_model, robot_data)
        for _ in range(np.random.randint(10, 60)):
            mujoco.mj_step(robot_model, robot_data)

        renderer_r.update_scene(robot_data, camera="overhead_cam")
        rgb_img = renderer_r.render()

        seg_renderer_r.update_scene(robot_data, camera="overhead_cam")
        seg_img = seg_renderer_r.render()

        img_h, img_w = rgb_img.shape[:2]
        if i == frames_to_generate - 1:
            vis_img_r = rgb_img.copy()

        label_file = open(f"{DATASET_DIR_R}/labels/train/frame_{i}.txt", "w")

        for obj_name, class_idx in classes_r.items():
            body_id = mujoco.mj_name2id(robot_model, mujoco.mjtObj.mjOBJ_BODY, obj_name)
            geom_start = robot_model.body_geomadr[body_id]
            geom_num = robot_model.body_geomnum[body_id]
            mask = np.isin(seg_img[:, :, 0], range(geom_start, geom_start + geom_num)).astype(np.uint8) * 255
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if contours:
                c = max(contours, key=cv2.contourArea)
                if cv2.contourArea(c) > 50:
                    rect = cv2.minAreaRect(c)
                    box = cv2.boxPoints(rect)
                    if vis_img_r is not None:
                        cv2.drawContours(vis_img_r, [np.int32(box)], 0, (0, 255, 0), 2)
                    box[:, 0] /= img_w
                    box[:, 1] /= img_h
                    label_str = " ".join([f"{coord:.5f}" for coordinate in box for coord in coordinate])
                    label_file.write(f"{class_idx} {label_str}\n")

        label_file.close()
        cv2.imwrite(f"{DATASET_DIR_R}/images/train/frame_{i}.jpg",
                    cv2.cvtColor(rgb_img, cv2.COLOR_RGB2BGR))

    print("Dataset generation complete!")
    if vis_img_r is not None:
        media.show_image(vis_img_r)

    # Write YAML and retrain
    yaml_config_r = {
        "path": os.path.abspath(DATASET_DIR_R),
        "train": "images/train",
        "val":   "images/train",
        "names": {0: "object", 1: "broom"},
    }
    with open("mujoco_dataset_with_broom.yaml", "w") as f:
        yaml.dump(yaml_config_r, f)

    model_obb_r = YOLO("yolov8n-obb.pt")
    results_r = model_obb_r.train(
        data="mujoco_dataset_with_broom.yaml",
        epochs=50, imgsz=640, device="cpu", verbose=False,
        lr0=0.001, batch=16
    )
    print("Training finished!")

    # Automatically resolve the correct weights path from the completed training run
    BROOM_WEIGHTS = str(results_r.save_dir / "weights" / "best.pt")
    print(f"BROOM_WEIGHTS: {BROOM_WEIGHTS}")

# 6. Grasp Planning

In [ ]:
import time

PANDA_READY = np.array([0, -np.pi / 4, 0, -3 * np.pi / 4, 0, np.pi / 2, np.pi / 4], dtype=np.float32)

# --- Joint limits for IK projection ---
def _get_joint_limits(model, joint_names):
    limits = []
    for name in joint_names:
        j_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name)
        lo, hi = model.jnt_range[j_id]
        if lo == 0.0 and hi == 0.0:
            lo, hi = -3.0, 3.0
        limits.append((float(lo), float(hi)))
    return np.array(limits, dtype=np.float32)

JOINT_LIMITS = _get_joint_limits(robot_model, PANDA_JOINTS)

def _clip_q(q, limits):
    return np.minimum(np.maximum(q, limits[:, 0]), limits[:, 1])

# --- Reset scene to model defaults ---
def reset_scene(model, data, arm_qpos=None):
    mujoco.mj_resetData(model, data)
    data.qpos[:] = model.qpos0
    data.qvel[:] = 0.0
    if model.nu > 0:
        data.ctrl[:] = 0.0
    if arm_qpos is not None:
        jnt_ids = [mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, n) for n in PANDA_JOINTS]
        qpos_adr = [model.jnt_qposadr[j] for j in jnt_ids]
        for i, adr in enumerate(qpos_adr):
            data.qpos[adr] = arm_qpos[i]
    mujoco.mj_forward(model, data)

# --- Simple settle step to let objects fall ---
def settle_scene(model, data, arm_qpos=None, steps=200):
    jnt_ids = [mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, n) for n in PANDA_JOINTS]
    qpos_adr = [model.jnt_qposadr[j] for j in jnt_ids]
    dof_adr = [model.jnt_dofadr[j] for j in jnt_ids]
    for _ in range(steps):
        if arm_qpos is not None:
            for i, adr in enumerate(qpos_adr):
                data.qpos[adr] = arm_qpos[i]
            for d in dof_adr:
                data.qvel[d] = 0.0
        mujoco.mj_step(model, data)
    mujoco.mj_forward(model, data)

# --- Position + orientation IK (damped least squares) ---
def solve_ik_to_site(
    model,
    data,
    site_name,
    target_pos,
    q_init,
    target_rot=None,
    max_iters=400,
    tol=2e-3,
    rot_tol=5e-2,
    damp=5e-3,
    rot_w=0.2,
 ):
    jnt_ids = [mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, n) for n in PANDA_JOINTS]
    qpos_adr = [model.jnt_qposadr[j] for j in jnt_ids]
    dof_adr = [model.jnt_dofadr[j] for j in jnt_ids]
    site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, site_name)

    target_pos = target_pos.copy()
    target_pos[2] = max(target_pos[2], 0.05)

    q = q_init.copy()
    for _ in range(max_iters):
        q = _clip_q(q, JOINT_LIMITS)
        for i, adr in enumerate(qpos_adr):
            data.qpos[adr] = q[i]
        mujoco.mj_forward(model, data)

        cur = data.site_xpos[site_id].copy()
        err_pos = target_pos - cur
        if np.linalg.norm(err_pos) > 0.1:
            err_pos = err_pos / np.linalg.norm(err_pos) * 0.1

        jacp = np.zeros((3, model.nv))
        jacr = np.zeros((3, model.nv))
        mujoco.mj_jacSite(model, data, jacp, jacr, site_id)

        rot_err = None
        if target_rot is not None:
            cur_mat = data.site_xmat[site_id].reshape(3, 3)
            quat_cur = np.zeros(4, dtype=np.float64)
            quat_tgt = np.zeros(4, dtype=np.float64)
            mujoco.mju_mat2Quat(quat_cur, cur_mat.flatten())
            mujoco.mju_mat2Quat(quat_tgt, target_rot.flatten())
            quat_err = np.zeros(4, dtype=np.float64)
            quat_cur_conj = quat_cur.copy()
            quat_cur_conj[1:] *= -1.0
            mujoco.mju_mulQuat(quat_err, quat_tgt, quat_cur_conj)
            rot_err = np.zeros(3, dtype=np.float64)
            mujoco.mju_quat2Vel(rot_err, quat_err, 1.0)
            if np.linalg.norm(rot_err) > 0.5:
                rot_err = rot_err / np.linalg.norm(rot_err) * 0.5

        if np.linalg.norm(err_pos) < tol and (rot_err is None or np.linalg.norm(rot_err) < rot_tol):
            return q, True

        if rot_err is not None:
            err = np.concatenate([err_pos, rot_w * rot_err])
            J = np.vstack([jacp[:, dof_adr], rot_w * jacr[:, dof_adr]])
            A = J @ J.T + damp * np.eye(6)
        else:
            err = err_pos
            J = jacp[:, dof_adr]
            A = J @ J.T + damp * np.eye(3)

        dq = J.T @ np.linalg.solve(A, err)
        q = q + 0.5 * dq

    return q, False

def _get_arm_actuator_ids(model, joint_names):
    jnt_ids = [mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, n) for n in joint_names]
    jnt_to_idx = {jid: i for i, jid in enumerate(jnt_ids)}
    act_ids = []
    for act_id in range(model.nu):
        joint_id = int(model.actuator_trnid[act_id, 0])
        if joint_id in jnt_to_idx:
            act_ids.append((jnt_to_idx[joint_id], act_id))
    act_ids.sort(key=lambda x: x[0])
    return [act_id for _, act_id in act_ids], jnt_ids

def _find_gripper_actuators_by_name(model):
    act_ids = []
    for act_id in range(model.nu):
        act_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, act_id)
        if act_name and ("finger" in act_name or "gripper" in act_name):
            act_ids.append(act_id)
    return act_ids

def _find_gripper_actuators_by_tendon(model):
    act_ids = []
    for act_id in range(model.nu):
        if int(model.actuator_trntype[act_id]) != int(mujoco.mjtTrn.mjTRN_TENDON):
            continue
        tendon_id = int(model.actuator_trnid[act_id, 0])
        tendon_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_TENDON, tendon_id)
        if tendon_name and ("finger" in tendon_name or "gripper" in tendon_name or "split" in tendon_name):
            act_ids.append(act_id)
    return act_ids

def _get_gripper_actuator_ids(model, joint_names):
    hardcoded = ["panda_actuator8"]
    hardcoded_ids = [
        mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, name)
        for name in hardcoded
    ]
    if all(aid >= 0 for aid in hardcoded_ids):
        return hardcoded_ids

    tendon_act_ids = _find_gripper_actuators_by_tendon(model)
    if tendon_act_ids:
        return tendon_act_ids

    jnt_ids = [mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, n) for n in joint_names]
    jnt_to_idx = {jid: i for i, jid in enumerate(jnt_ids)}
    act_ids = []
    for act_id in range(model.nu):
        joint_id = int(model.actuator_trnid[act_id, 0])
        if joint_id in jnt_to_idx:
            act_ids.append((jnt_to_idx[joint_id], act_id))
    act_ids.sort(key=lambda x: x[0])
    joint_act_ids = [act_id for _, act_id in act_ids]
    if joint_act_ids:
        return joint_act_ids

    name_act_ids = _find_gripper_actuators_by_name(model)
    if name_act_ids:
        return name_act_ids

    # Fall back to last two actuators (Panda gripper in menagerie).
    if model.nu >= 2:
        return [model.nu - 2, model.nu - 1]
    return []

def _set_gripper_qpos(model, data, joint_names, value):
    for name in joint_names:
        j_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name)
        if j_id < 0:
            continue
        adr = model.jnt_qposadr[j_id]
        data.qpos[adr] = float(value)
    mujoco.mj_forward(model, data)

def _debug_actuators(model, act_ids):
    info = []
    for act_id in act_ids:
        act_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, act_id)
        joint_id = int(model.actuator_trnid[act_id, 0])
        joint_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, joint_id)
        gear = float(model.actuator_gear[act_id, 0])
        lo, hi = model.actuator_forcerange[act_id]
        info.append((act_id, act_name, joint_name, gear, (float(lo), float(hi))))
    print("Actuators driving arm:", info)
    print("Total actuators:", model.nu)

def _set_torque_limits(model, act_ids, arm_limits, wrist_limits):
    for i, act_id in enumerate(act_ids):
        limit = arm_limits if i < 4 else wrist_limits
        model.actuator_forcerange[act_id, 0] = -float(limit)
        model.actuator_forcerange[act_id, 1] = float(limit)
    print("Updated actuator forcerange:", [tuple(model.actuator_forcerange[a]) for a in act_ids])

def _apply_direct_torque(data, dof_adr, tau):
    data.qfrc_applied[:] = 0.0
    for i, d in enumerate(dof_adr):
        data.qfrc_applied[d] = float(tau[i])


# --- Execute joint path with PD torque control ---
def execute_joint_path(
    model,
    data,
    path,
    renderer,
    camera="overhead_cam",
    hold_steps=8,
    sleep_s=0.0,
    kp=800.0,
    kd=30.0,
    wait_on_exit=True, # Kept for signature compatibility, but ignored now
    exit_key="q",      # Kept for signature compatibility, but ignored now
    target_pos=None,
    use_direct_torque=False,
    render_stride=3,
    gripper_ctrl=None,
    reset_start_pose=True,
    frames_out=None    # NEW: Pass a list to collect video frames
):
    if path is None or len(path) == 0:
        print("No path to execute.")
        return

    act_ids, jnt_ids = _get_arm_actuator_ids(model, PANDA_JOINTS)
    if not act_ids:
        print("No arm actuators found; cannot run torque control.")
        return
    _debug_actuators(model, act_ids)
    _set_torque_limits(model, act_ids, arm_limits=200.0, wrist_limits=80.0)
    gripper_act_ids = _get_gripper_actuator_ids(model, FINGER_JOINTS)

    qpos_adr = [model.jnt_qposadr[j] for j in jnt_ids]
    dof_adr = [model.jnt_dofadr[j] for j in jnt_ids]
    print("Arm dof adr:", dof_adr)

    def _apply_gripper_ctrl():
        if gripper_ctrl is None or not gripper_act_ids:
            return
        for act_id in gripper_act_ids:
            data.ctrl[act_id] = float(gripper_ctrl)

    if reset_start_pose:
        for i, adr in enumerate(qpos_adr):
            data.qpos[adr] = path[0][i]
        for d in dof_adr:
            data.qvel[d] = 0.0
        data.ctrl[:] = 0.0
        _apply_gripper_ctrl()
        mujoco.mj_forward(model, data)
    else:
        _apply_gripper_ctrl()
        mujoco.mj_forward(model, data)

    # Hold the start pose briefly to confirm torque control is active
    for _ in range(30):
        q = np.array([data.qpos[a] for a in qpos_adr], dtype=np.float32)
        qd = np.array([data.qvel[a] for a in dof_adr], dtype=np.float32)
        tau = kp * (path[0] - q) - kd * qd + data.qfrc_bias[dof_adr]
        if use_direct_torque:
            _apply_direct_torque(data, dof_adr, tau)
        else:
            for i, act_id in enumerate(act_ids):
                lo, hi = model.actuator_forcerange[act_id]
                data.ctrl[act_id] = float(np.clip(tau[i], lo, hi))
        _apply_gripper_ctrl()
        mujoco.mj_step(model, data)
    print("Hold max |tau|:", float(np.max(np.abs(tau))))
    mujoco.mj_forward(model, data)

    render_count = 0
    for q_target in path:
        for _ in range(hold_steps):
            q = np.array([data.qpos[a] for a in qpos_adr], dtype=np.float32)
            qd = np.array([data.qvel[a] for a in dof_adr], dtype=np.float32)
            tau = kp * (q_target - q) - kd * qd + data.qfrc_bias[dof_adr]
            if use_direct_torque:
                _apply_direct_torque(data, dof_adr, tau)
            else:
                for i, act_id in enumerate(act_ids):
                    lo, hi = model.actuator_forcerange[act_id]
                    data.ctrl[act_id] = float(np.clip(tau[i], lo, hi))
            _apply_gripper_ctrl()
            mujoco.mj_step(model, data)
            render_count += 1
            
            if render_count % render_stride != 0:
                continue
            
            renderer.update_scene(data, camera=camera)

            if target_pos is not None:
                scene = renderer.scene
                if scene.ngeom < scene.maxgeom:
                    mujoco.mjv_initGeom(
                        scene.geoms[scene.ngeom],
                        mujoco.mjtGeom.mjGEOM_SPHERE, np.array([0.05, 0, 0]),
                        target_pos, np.eye(3).flatten(), np.array([1, 0, 1, 0.7], dtype=np.float32)
                    )
                    scene.geoms[scene.ngeom].category = mujoco.mjtCatBit.mjCAT_DECOR
                    scene.ngeom += 1

            # Capture frame instead of showing via cv2
            rgb = renderer.render()
            if frames_out is not None:
                frames_out.append(rgb.copy())

In [ ]:
def _pick_broom_detection(detections):
    # Prefer the broom class when present; otherwise return the most confident object.
    if not detections:
        return None
    broom_dets = [d for d in detections if d.get("class") == "broom"]
    candidates = broom_dets if broom_dets else detections
    return max(candidates, key=lambda d: float(d.get("confidence", 0.0)))

In [ ]:
def _camera_to_world(model, cam_name, pos_cam):
    cam_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_CAMERA, cam_name)
    cam_pos = model.cam_pos[cam_id].copy()
    cam_quat = model.cam_quat[cam_id].copy()
    cam_mat = np.zeros(9, dtype=np.float64)
    mujoco.mju_quat2Mat(cam_mat, cam_quat)
    cam_R = cam_mat.reshape(3, 3)
    return cam_pos + cam_R @ pos_cam

In [ ]:
# --- IK demo: use perceived broom handle as target ---
if "perceptor_broom" not in globals():
    perceptor_broom = PerceptionModule(BROOM_WEIGHTS)

arm_seed = PANDA_READY
reset_scene(robot_model, robot_data, arm_qpos=arm_seed)

# Set broom to an absolute world position (preserve current z).
broom_body_id = mujoco.mj_name2id(robot_model, mujoco.mjtObj.mjOBJ_BODY, "broom")
broom_jnt_adr = robot_model.jnt_qposadr[robot_model.body_jntadr[broom_body_id]]
broom_target_xy = np.array([0.05, -0.25], dtype=np.float32)
broom_z = float(robot_data.qpos[broom_jnt_adr + 2])
robot_data.qpos[broom_jnt_adr + 0] = float(broom_target_xy[0])
robot_data.qpos[broom_jnt_adr + 1] = float(broom_target_xy[1])
robot_data.qpos[broom_jnt_adr + 2] = broom_z

settle_scene(robot_model, robot_data, arm_qpos=arm_seed, steps=200)

# Open gripper before perception and motion.
gripper_act_ids = _get_gripper_actuator_ids(robot_model, FINGER_JOINTS)
gripper_open = 0.04
gripper_closed = 0.0
gripper_available = bool(gripper_act_ids)
if gripper_available:
    ctrl_lo, ctrl_hi = robot_model.actuator_ctrlrange[gripper_act_ids[0]]
    gripper_closed = float(ctrl_lo)
    gripper_open = float(ctrl_hi)
    gripper_preopen = gripper_open * 0.6
    
    # Boost gripper grip strength to prevent slippage
    for act_id in gripper_act_ids:
        robot_model.actuator_forcerange[act_id, 0] = -500.0
        robot_model.actuator_forcerange[act_id, 1] = 500.0
        robot_model.actuator_biasprm[act_id, 1] = -5000.0
        robot_model.actuator_biasprm[act_id, 2] = -50.0
        if ctrl_hi > 0:
            # We doubled the joint range in the spec, so we need to match the gain mapping
            robot_model.actuator_gainprm[act_id, 0] = 0.06 * 5000.0 / ctrl_hi

    gripper_act_names = [
        mujoco.mj_id2name(robot_model, mujoco.mjtObj.mjOBJ_ACTUATOR, act_id)
        for act_id in gripper_act_ids
    ]
    print("Gripper actuators:", gripper_act_names)
    print("Gripper ctrlrange:", robot_model.actuator_ctrlrange[gripper_act_ids])
    for _ in range(30):
        for act_id in gripper_act_ids:
            robot_data.ctrl[act_id] = float(gripper_preopen)
        mujoco.mj_step(robot_model, robot_data)
else:
    all_act_names = [
        mujoco.mj_id2name(robot_model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
        for i in range(robot_model.nu)
    ]
    print("Warning: no gripper actuators found; cannot open gripper.")
    print("Available actuators:", all_act_names)

def _rotation_from_obb(rotation_rad):
    c = float(np.cos(rotation_rad))
    s = float(np.sin(rotation_rad))
    x_axis = np.array([c, s, 0.0], dtype=np.float32)
    z_axis = np.array([0.0, 0.0, -1.0], dtype=np.float32)
    y_axis = np.cross(z_axis, x_axis)
    x_axis /= np.linalg.norm(x_axis) + 1e-8
    y_axis /= np.linalg.norm(y_axis) + 1e-8
    z_axis = np.cross(x_axis, y_axis)
    return np.stack([x_axis, y_axis, z_axis], axis=1)


def _rotation_z_offset(rad):
    c = float(np.cos(rad))
    s = float(np.sin(rad))
    return np.array([
        [c, -s, 0.0],
        [s,  c, 0.0],
        [0.0, 0.0, 1.0],
    ], dtype=np.float32)


def _camera_to_world_rotation(model, cam_name, rot_cam):
    cam_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_CAMERA, cam_name)
    cam_quat = model.cam_quat[cam_id].copy()
    cam_mat = np.zeros(9, dtype=np.float64)
    mujoco.mju_quat2Mat(cam_mat, cam_quat)
    cam_R = cam_mat.reshape(3, 3)
    return cam_R @ rot_cam


def _rotation_y_90():
    return np.array([
        [0.0, 0.0, 1.0],
        [0.0, 1.0, 0.0],
        [-1.0, 0.0, 0.0],
    ], dtype=np.float32)

renderer_t = mujoco.Renderer(robot_model, 480, 640)
depth_renderer_t = mujoco.Renderer(robot_model, 480, 640)
depth_renderer_t.enable_depth_rendering()

renderer_t.update_scene(robot_data, camera="overhead_cam")
depth_renderer_t.update_scene(robot_data, camera="overhead_cam")
img_rgb = renderer_t.render()
img_depth = depth_renderer_t.render()

detected_objects, _ = perceptor_broom.get_transforms(img_rgb, img_depth)
broom_det = _pick_broom_detection(detected_objects)

if broom_det is None:
    print("No broom detected; check weights or camera framing.")
else:
    broom_handle_cam = broom_det["position_camera_frame"].astype(np.float32)
    broom_handle_world = _camera_to_world(robot_model, "overhead_cam", broom_handle_cam)

    # Toggle reflections to debug axis mirroring.
    mirror_y = True
    mirror_z_about_cam = True

    cam_id = mujoco.mj_name2id(robot_model, mujoco.mjtObj.mjOBJ_CAMERA, "overhead_cam")
    cam_pos = robot_model.cam_pos[cam_id].copy()
    if mirror_y:
        broom_handle_world[1] = -broom_handle_world[1]
    if mirror_z_about_cam:
        broom_handle_world[2] = 2.0 * cam_pos[2] - broom_handle_world[2]

    rot_cam = broom_det.get("rotation_camera_frame")
    if rot_cam is None:
        rot_cam = _rotation_from_obb(broom_det.get("rotation_rad", 0.0))
    rot_cam = rot_cam @ _rotation_z_offset(-np.pi / 4.0)
    rot_world = _camera_to_world_rotation(robot_model, "overhead_cam", rot_cam)
    if mirror_y:
        rot_world = np.diag([1.0, -1.0, 1.0]).astype(np.float32) @ rot_world
    if mirror_z_about_cam:
        rot_world = np.diag([1.0, 1.0, -1.0]).astype(np.float32) @ rot_world

    rot_world = rot_world @ _rotation_y_90()

    print("Perceived handle (world):", broom_handle_world)

    approach_offset = np.array([0.0, 0.08, 0.0], dtype=np.float32)
    target_pos = broom_handle_world.copy()
    target_rot = rot_world.copy()
    grasp_lateral_offset = 0.015
    # Nudge along the gripper width axis to avoid the finger gap.
    target_pos = target_pos + target_rot[:, 1] * grasp_lateral_offset
    pregrasp_pos = target_pos + approach_offset
    site_id = mujoco.mj_name2id(robot_model, mujoco.mjtObj.mjOBJ_SITE, EE_SITE)
    gripper_pos = robot_data.site_xpos[site_id].copy()
    print("Gripper pos (world):", gripper_pos)

    
    # ... (Keep everything above this point the same, down to the q_start definition) ...
    q_start = np.array([
        robot_data.qpos[robot_model.jnt_qposadr[mujoco.mj_name2id(
            robot_model, mujoco.mjtObj.mjOBJ_JOINT, n
)]] for n in PANDA_JOINTS
    ])
    
    if gripper_available:
        for _ in range(20):
            for act_id in gripper_act_ids:
                robot_data.ctrl[act_id] = float(gripper_open)
            mujoco.mj_step(robot_model, robot_data)

    video_frames = [] # NEW: Initialize an empty list to store our video

    q_pre, ok_pre = solve_ik_to_site(robot_model, robot_data, EE_SITE, pregrasp_pos, q_start, target_rot=target_rot)
    if not ok_pre:
        print("IK demo: failed to reach pregrasp; try a different pose")
    else:
        steps = 30
        path_pre = [(1.0 - t) * q_start + t * q_pre for t in np.linspace(0.0, 1.0, steps)]
        ik_renderer = mujoco.Renderer(robot_model, 480, 640)
        execute_joint_path(
            robot_model, robot_data, path_pre, ik_renderer,
            target_pos=pregrasp_pos, use_direct_torque=True, render_stride=3,
            gripper_ctrl=gripper_open if gripper_available else None,
            frames_out=video_frames # NEW: Pass the list
        )

        q_goal, ok = solve_ik_to_site(robot_model, robot_data, EE_SITE, target_pos, q_pre, target_rot=target_rot)
        if ok:
            steps = 20
            path = [(1.0 - t) * q_pre + t * q_goal for t in np.linspace(0.0, 1.0, steps)]
            execute_joint_path(
                robot_model, robot_data, path, ik_renderer,
                target_pos=target_pos, use_direct_torque=True, render_stride=3,
                gripper_ctrl=gripper_open if gripper_available else None,
                frames_out=video_frames # NEW: Pass the list
            )

            if gripper_available:
                print("Closing gripper...")
                execute_joint_path(
                    robot_model, robot_data, [q_goal], ik_renderer,
                    target_pos=target_pos, use_direct_torque=True, render_stride=10,
                    gripper_ctrl=gripper_closed, hold_steps=50, 
                    frames_out=video_frames # NEW: Pass the list
                )
                for _ in range(50):
                    for act_id in gripper_act_ids:
                        robot_data.ctrl[act_id] = float(gripper_closed)
                    mujoco.mj_step(robot_model, robot_data)
                print("Gripper closed!")
                
                print("Attempting to lift broom to test friction...")
                lift_pos = target_pos + np.array([0.0, 0.0, 0.3], dtype=np.float32)
                q_lift, ok_lift = solve_ik_to_site(robot_model, robot_data, EE_SITE, lift_pos, q_goal, target_rot=target_rot)
                
                if ok_lift:
                    steps_lift = 40
                    path_lift = [(1.0 - t) * q_goal + t * q_lift for t in np.linspace(0.0, 1.0, steps_lift)]
                    execute_joint_path(
                        robot_model, robot_data, path_lift, ik_renderer,
                        target_pos=lift_pos, use_direct_torque=True, render_stride=3,
                        gripper_ctrl=gripper_closed, 
                        frames_out=video_frames # NEW: Pass the list
                    )
                    print("Lift complete! Holding...")
                    
                    execute_joint_path(
                        robot_model, robot_data, [q_lift], ik_renderer,
                        target_pos=lift_pos, use_direct_torque=True, render_stride=10,
                        gripper_ctrl=gripper_closed, hold_steps=100, # Reduced hold_steps so video doesn't just sit there forever
                        reset_start_pose=False,
                        frames_out=video_frames # NEW: Pass the list
                    )
                else:
                    print("IK failed for lift.")
            else:
                print("Warning: no gripper actuators found; cannot close gripper.")
        else:
            print("IK demo: failed to reach perceived target; try a different pose")

    # NEW: Display the combined video inline using mediapy
    import mediapy as media
    if len(video_frames) > 0:
        print(f"Rendering video with {len(video_frames)} frames...")
        media.show_video(video_frames, fps=30)

In [ ]:
tendons = [
    (i, mujoco.mj_id2name(robot_model, mujoco.mjtObj.mjOBJ_TENDON, i))
    for i in range(robot_model.ntendon)
]
print("Tendons:", tendons)

act_tendons = []
for act_id in range(robot_model.nu):
    if int(robot_model.actuator_trntype[act_id]) == int(mujoco.mjtTrn.mjTRN_TENDON):
        t_id = int(robot_model.actuator_trnid[act_id, 0])
        act_tendons.append((
            act_id,
            mujoco.mj_id2name(robot_model, mujoco.mjtObj.mjOBJ_ACTUATOR, act_id),
            t_id,
            mujoco.mj_id2name(robot_model, mujoco.mjtObj.mjOBJ_TENDON, t_id),
            robot_model.actuator_ctrlrange[act_id]
        ))
print("Tendon actuators:", act_tendons)

# 7. Reinforcement Learning for Sweeping
We define a goal area and a reward function based on the distance of all objects (mug, can_opener, action_fig, shoe) to that goal.

In [ ]:
# --- NEW CELL 1: SETUP RL ON TOP OF EXISTING DATA ---

rl_model = robot_model

rl_data = mujoco.MjData(rl_model)
rl_data.qpos[:] = robot_data.qpos.copy()
rl_data.qvel[:] = robot_data.qvel.copy()
rl_data.ctrl[:] = robot_data.ctrl.copy()
mujoco.mj_forward(rl_model, rl_data)

# Goal region — centred in camera frame, in the robot's natural sweep direction
# Camera sees X ∈ [−0.83, 0.83], Y ∈ [−0.62, 0.62] at table level.
GOAL_CENTER = np.array([-0.2, 0.0], dtype=np.float32)
GOAL_RADIUS = 0.15

cam_half_x, cam_half_y = 0.828, 0.621
assert abs(GOAL_CENTER[0]) + GOAL_RADIUS < cam_half_x, "Goal clips camera X!"
assert abs(GOAL_CENTER[1]) + GOAL_RADIUS < cam_half_y, "Goal clips camera Y!"
print(f"Goal region passes camera bounds check.")

GRIPPER_ACT_IDX     = 7
GRIPPER_CLOSED_CTRL = float(rl_data.ctrl[GRIPPER_ACT_IDX]) if rl_model.nu > 7 else None

# ── Capture the end-effector pose at grasp time ───────────────────────────
# This is the orientation the controller will defend throughout the episode.
# Reading it directly from the current simulation state means it exactly
# matches whatever the IK stage converged to — no manual matrix needed.
_site_id = mujoco.mj_name2id(rl_model, mujoco.mjtObj.mjOBJ_SITE, EE_SITE)
EE_ROT_FIXED = rl_data.site_xmat[_site_id].copy().reshape(3, 3)  # (3,3) world-frame
EE_POS_INIT  = rl_data.site_xpos[_site_id].copy()                # (3,)  world-frame

print(f"Captured EE grasp pose:")
print(f"  position    : {EE_POS_INIT}")
print(f"  rotation row0: {EE_ROT_FIXED[0]}")

# ── Workspace bounds — keeps the broom on the table, arm in reachable zone ─
# Z bounds: keep the broom head near table level (don't lift or crash into table)
WS_LO = np.array([-0.50, -0.55,  EE_POS_INIT[2] - 0.05], dtype=np.float32)
WS_HI = np.array([ 0.55,  0.55,  EE_POS_INIT[2] + 0.08], dtype=np.float32)

rl_init_qpos = rl_data.qpos.copy()
rl_init_qvel = rl_data.qvel.copy()
rl_init_ctrl = rl_data.ctrl.copy()

OBJ_NAMES = ["mug", "can_opener", "action_fig", "shoe"]
OBJ_IDS   = [mujoco.mj_name2id(rl_model, mujoco.mjtObj.mjOBJ_BODY, n) for n in OBJ_NAMES]
BROOM_ID  = mujoco.mj_name2id(rl_model, mujoco.mjtObj.mjOBJ_BODY, "broom")

print(f"\nRL setup complete. Gripper ctrl={GRIPPER_CLOSED_CTRL:.4f}")
print(f"Workspace: lo={WS_LO}, hi={WS_HI}")

In [ ]:
# ── STANDALONE REWARD FUNCTION ────────────────────────────────────────────
# Defined outside SweeperEnv so the test demo can call it directly
# without needing a running MuJoCo simulation.

def compute_reward(broom_xy, obj_positions, swept_mask, goal_center, goal_radius):
    """
    Four-metric reward function for the sweeping task.

    Metric 1 (W=1.0): Negative mean distance of ALL objects to goal centre.
                       Closer objects → less negative → higher reward.

    Metric 2 (W=10.0): +10 per-step for each object currently inside the
                        goal region. Creates a large, visible reward spike
                        the moment an object crosses the boundary.

    Metric 3 (W=0.30): Negative distance from broom to the NEAREST unswept
                        object only. Once an object is swept the broom is
                        pulled toward the next one automatically.

    Metric 4 (W=0.60): Angle reward — measures how well the broom is
                        positioned BEHIND the nearest unswept object.

                        The triangle is:  BROOM ── [OBJECT] ── GOAL
                        The angle is measured at the OBJECT vertex between
                        the vectors (O→broom) and (O→goal).

                          180° (cos=-1): broom directly behind object → +W4
                           90° (cos= 0): broom to the side              →  0
                            0° (cos=+1): broom between object and goal  → -W4
                                         + extra penalty if also close

                        W4 > W3 so good positioning outweighs proximity.

    Returns: (total_reward, breakdown_dict)
    """
    W1, W2, W3, W4    = 1.0, 10.0, 0.30, 0.60
    TOO_CLOSE_THRESH  = 0.12   # metres: wrong-side proximity penalty radius

    broom_xy    = np.array(broom_xy,    dtype=np.float64)
    goal_center = np.array(goal_center, dtype=np.float64)
    n           = len(obj_positions)

    # ── Metric 1: average object-to-goal distance ─────────────────────────
    distances_to_goal = [
        float(np.linalg.norm(np.array(p, dtype=np.float64) - goal_center))
        for p in obj_positions
    ]
    r1 = -W1 * float(np.mean(distances_to_goal))

    # ── Metric 2: swept bonus ─────────────────────────────────────────────
    n_swept = int(np.sum(swept_mask))
    r2      = W2 * n_swept

    # ── Metrics 3 & 4: broom vs nearest unswept object ───────────────────
    r3, r4       = 0.0, 0.0
    angle_deg    = None
    nearest_idx  = None

    unswept_indices = [i for i in range(n) if not swept_mask[i]]

    if unswept_indices:
        # Metric 3: find nearest unswept object to broom
        dists_to_broom = [
            float(np.linalg.norm(
                np.array(obj_positions[i], dtype=np.float64) - broom_xy
            ))
            for i in unswept_indices
        ]
        local_nearest = int(np.argmin(dists_to_broom))
        nearest_idx   = unswept_indices[local_nearest]
        nearest_pos   = np.array(obj_positions[nearest_idx], dtype=np.float64)
        nearest_dist  = dists_to_broom[local_nearest]

        r3 = -W3 * nearest_dist

        # Metric 4: angle at object vertex
        #
        #   Desired layout:   BROOM ──→ [OBJECT] ──→ GOAL
        #
        #   vec_O_to_B: vector from object toward broom
        #   vec_O_to_G: vector from object toward goal
        #
        #   When these point in OPPOSITE directions (cos=-1, angle=180°):
        #   broom is directly behind the object — perfect sweep position.
        #
        #   When they point in the SAME direction (cos=+1, angle=0°):
        #   broom has crossed to the goal side — wrong position.

        vec_O_to_B = broom_xy    - nearest_pos
        vec_O_to_G = goal_center - nearest_pos

        d_OB = float(np.linalg.norm(vec_O_to_B)) + 1e-8
        d_OG = float(np.linalg.norm(vec_O_to_G)) + 1e-8

        cos_angle = float(np.dot(vec_O_to_B / d_OB, vec_O_to_G / d_OG))
        cos_angle = float(np.clip(cos_angle, -1.0, 1.0))
        angle_deg = float(np.degrees(np.arccos(cos_angle)))

        # -cos maps: cos=-1 (180°,perfect) → +1, cos=+1 (0°,wrong) → -1
        r4 = W4 * (-cos_angle)

        # Extra penalty: broom on WRONG side (cos>0) AND uncomfortably close
        if cos_angle > 0.0 and nearest_dist < TOO_CLOSE_THRESH:
            wrong_side_penalty = (
                W4 * cos_angle
                * (1.0 - nearest_dist / TOO_CLOSE_THRESH)
            )
            r4 -= wrong_side_penalty

    total = r1 + r2 + r3 + r4

    return total, {
        "r1_obj_dist":       r1,
        "r2_swept_bonus":    r2,
        "r3_broom_prox":     r3,
        "r4_angle":          r4,
        "total":             total,
        "n_swept":           n_swept,
        "distances_to_goal": distances_to_goal,
        "angle_deg":         angle_deg,
        "nearest_idx":       nearest_idx,
    }

print("compute_reward() defined.")

In [ ]:
# --- NEW CELL 2: RL ENVIRONMENT ---
import gymnasium as gym
from gymnasium import spaces

class SweeperEnv(gym.Env):
    MAX_STEPS     = 500
    EE_STEP_SIZE  = 0.04
    TORQUE_LIMITS = np.array([87, 87, 87, 87, 12, 12, 12], dtype=np.float32)

    def __init__(self, model, init_qpos, init_qvel, init_ctrl,
                 goal_center, goal_radius,
                 obj_ids, broom_id, gripper_ctrl,
                 ee_rot_fixed, ee_pos_init, ws_lo, ws_hi):
        super().__init__()
        self.model         = model
        self.data          = mujoco.MjData(model)
        self.init_qpos     = init_qpos.copy()
        self.init_qvel     = init_qvel.copy()
        self.init_ctrl     = init_ctrl.copy()
        self.goal_center   = np.array(goal_center, dtype=np.float32)
        self.goal_radius   = float(goal_radius)
        self.obj_ids       = list(obj_ids)
        self.broom_id      = int(broom_id)
        self.gripper_ctrl  = float(gripper_ctrl) if gripper_ctrl is not None else None
        self._ee_rot_fixed = ee_rot_fixed.astype(np.float64)
        self._ws_lo        = ws_lo.astype(np.float64)
        self._ws_hi        = ws_hi.astype(np.float64)
        self._step_count   = 0

        self.action_space = spaces.Box(-1.0, 1.0, shape=(2,), dtype=np.float32)
        n_obj   = len(self.obj_ids)
        obs_dim = 7 + 7 + n_obj * 2 + 2 + 2 + n_obj
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)

        self._site_id  = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, EE_SITE)
        self._arm_qadr = np.array([
            model.jnt_qposadr[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, jn)]
            for jn in PANDA_JOINTS
        ])
        self._arm_vadr = np.array([
            model.jnt_dofadr[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, jn)]
            for jn in PANDA_JOINTS
        ])

    # ── Jacobian controller (unchanged) ───────────────────────────────────
    def _delta_to_torques(self, delta_xy):
        nv      = self.model.nv
        site_id = self._site_id
        ee_pos  = self.data.site_xpos[site_id].copy()
        delta_xyz = np.array([delta_xy[0], delta_xy[1], 0.0], dtype=np.float64)
        target    = np.clip(ee_pos + delta_xyz, self._ws_lo, self._ws_hi)
        pos_err   = target - ee_pos

        R_cur   = self.data.site_xmat[site_id].reshape(3, 3).copy()
        R_err   = self._ee_rot_fixed @ R_cur.T
        angle   = np.arccos(np.clip((np.trace(R_err) - 1.0) / 2.0, -1.0, 1.0))
        if abs(angle) < 1e-6:
            rot_err = np.zeros(3)
        else:
            rot_err = (angle / (2.0 * np.sin(angle))) * np.array([
                R_err[2, 1] - R_err[1, 2],
                R_err[0, 2] - R_err[2, 0],
                R_err[1, 0] - R_err[0, 1],
            ])

        jacp = np.zeros((3, nv), dtype=np.float64)
        jacr = np.zeros((3, nv), dtype=np.float64)
        mujoco.mj_jac(self.model, self.data, jacp, jacr, ee_pos, site_id)

        Jp, Jr = jacp[:, self._arm_vadr], jacr[:, self._arm_vadr]
        ROT_W  = 0.3
        J6     = np.vstack([Jp,           ROT_W * Jr  ])
        err6   = np.concatenate([pos_err, ROT_W * rot_err])

        lam     = 1e-3
        dq      = J6.T @ np.linalg.solve(J6 @ J6.T + lam * np.eye(6), err6)
        kp, kd  = 300.0, 30.0
        torques = kp * dq - kd * self.data.qvel[self._arm_vadr]
        return np.clip(torques, -self.TORQUE_LIMITS, self.TORQUE_LIMITS)

    # ── Observation (unchanged) ───────────────────────────────────────────
    def _get_obs(self):
        arm_q    = self.data.qpos[self._arm_qadr].astype(np.float32)
        arm_v    = self.data.qvel[self._arm_vadr].astype(np.float32)
        obj_xy   = np.array(
            [self.data.xpos[oid][:2] for oid in self.obj_ids], dtype=np.float32
        ).ravel()
        broom_xy = self.data.xpos[self.broom_id][:2].astype(np.float32)
        dists    = np.array([
            np.linalg.norm(self.data.xpos[oid][:2] - self.goal_center)
            for oid in self.obj_ids
        ], dtype=np.float32)
        return np.concatenate([arm_q, arm_v, obj_xy, broom_xy, self.goal_center, dists])

    # ── Reward — delegates to standalone compute_reward() ─────────────────
    def _compute_reward(self):
        broom_xy      = self.data.xpos[self.broom_id][:2].copy()
        obj_positions = [self.data.xpos[oid][:2].copy() for oid in self.obj_ids]
        swept_mask    = [
            bool(np.linalg.norm(p - self.goal_center) < self.goal_radius)
            for p in obj_positions
        ]
        total, _ = compute_reward(
            broom_xy, obj_positions, swept_mask,
            self.goal_center, self.goal_radius
        )
        return total

    # ── Reset (unchanged) ─────────────────────────────────────────────────
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[:] = self.init_qpos
        self.data.qvel[:] = self.init_qvel
        self.data.ctrl[:] = self.init_ctrl
        mujoco.mj_forward(self.model, self.data)
        self._step_count = 0
        return self._get_obs(), {}

    # ── Step (unchanged) ──────────────────────────────────────────────────
    def step(self, action):
        self._step_count += 1
        delta_xy = action.astype(np.float64) * self.EE_STEP_SIZE
        self.data.ctrl[:7] = self._delta_to_torques(delta_xy)
        if self.gripper_ctrl is not None and self.model.nu > 7:
            self.data.ctrl[GRIPPER_ACT_IDX] = self.gripper_ctrl
        mujoco.mj_step(self.model, self.data)

        obs    = self._get_obs()
        reward = self._compute_reward()

        dists      = [np.linalg.norm(self.data.xpos[oid][:2] - self.goal_center)
                      for oid in self.obj_ids]
        terminated = bool(all(d < self.goal_radius for d in dists))
        truncated  = self._step_count >= self.MAX_STEPS
        if terminated:
            reward += 20.0

        return obs, reward, terminated, truncated, {}


rl_env = SweeperEnv(
    model=rl_model, init_qpos=rl_init_qpos, init_qvel=rl_init_qvel,
    init_ctrl=rl_init_ctrl, goal_center=GOAL_CENTER, goal_radius=GOAL_RADIUS,
    obj_ids=OBJ_IDS, broom_id=BROOM_ID, gripper_ctrl=GRIPPER_CLOSED_CTRL,
    ee_rot_fixed=EE_ROT_FIXED, ee_pos_init=EE_POS_INIT,
    ws_lo=WS_LO, ws_hi=WS_HI,
)
obs_test, _ = rl_env.reset()
print(f"Environment ready. obs_dim={rl_env.observation_space.shape[0]}, "
      f"act_dim={rl_env.action_space.shape[0]}")

In [ ]:
# ── REWARD FUNCTION TEST DEMO ─────────────────────────────────────────────
# Moves objects and broom through scripted scenarios to verify all 4 metrics.
# No MuJoCo simulation — positions are set directly.
import io
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.ioff()   # suppress inline display per-frame; we collect manually

# ── Config ────────────────────────────────────────────────────────────────
_GC  = np.array([-0.20,  0.00])   # goal centre (must match GOAL_CENTER)
_GR  = 0.15                        # goal radius
_OBJ_INIT = [                      # starting positions for the demo objects
    np.array([ 0.32,  0.38]),
    np.array([-0.38,  0.22]),
    np.array([ 0.22, -0.32]),
    np.array([-0.18, -0.40]),
]
_OBJ_COLORS = ['#ff5555', '#4488ff', '#ffaa00', '#cc55ff']

# ── Geometry helpers ──────────────────────────────────────────────────────
def _ease(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)

def _lerp(a, b, t):
    return np.array(a, float) + (np.array(b, float) - np.array(a, float)) * _ease(t)

def _behind(obj_pos, goal_center=_GC, dist=0.23):
    """Point directly behind obj_pos from goal_center's perspective."""
    d = np.array(obj_pos, float) - np.array(goal_center, float)
    return np.array(obj_pos, float) + d / (np.linalg.norm(d) + 1e-8) * dist

def _wrong_side(obj_pos, goal_center=_GC, dist=0.15):
    """Point between obj_pos and goal_center (wrong side for broom)."""
    d = np.array(goal_center, float) - np.array(obj_pos, float)
    return np.array(obj_pos, float) + d / (np.linalg.norm(d) + 1e-8) * dist

# ── Frame builder ─────────────────────────────────────────────────────────
def _build_demo_sequence():
    """
    Returns list of (broom_xy, obj_positions, swept_mask, phase_label).
    Objects move one by one to the goal; broom demonstrates each metric.
    """
    frames   = []
    obj_pos  = [p.copy() for p in _OBJ_INIT]
    swept    = [False, False, False, False]
    broom    = np.array([0.48, 0.0])

    def snapshot(label, n=1):
        sw = [np.linalg.norm(obj_pos[i] - _GC) < _GR for i in range(4)]
        for _ in range(n):
            frames.append((broom.copy(), [p.copy() for p in obj_pos], list(sw), label))

    def move_broom(target, n_frames, label):
        nonlocal broom
        start = broom.copy()
        for k in range(n_frames):
            broom = _lerp(start, target, k / max(n_frames - 1, 1))
            snapshot(label)
        broom = np.array(target, float)

    def sweep_obj(idx, n_frames, label):
        start = obj_pos[idx].copy()
        for k in range(n_frames):
            t = k / max(n_frames - 1, 1)
            obj_pos[idx] = _lerp(start, _GC, t)
            # Keep broom behind the moving object throughout
            nonlocal broom
            broom = _behind(obj_pos[idx])
            snapshot(label)
        obj_pos[idx] = _GC.copy()

    # ── Phase 0: initial state ────────────────────────────────────────────
    snapshot("Initial state — all 4 objects scattered, broom neutral", n=35)

    # ── Phase 1: demonstrate WRONG side (R4 penalty) ──────────────────────
    move_broom(_wrong_side(obj_pos[0]), 45,
               "Metric 4 demo — broom moves to WRONG side\n"
               "(between O1 and GOAL → angle < 90° → R4 penalised)")
    snapshot("Metric 4 demo — broom on wrong side (held)", n=25)

    # ── Phase 2: demonstrate CORRECT side (R4 reward) ─────────────────────
    move_broom(_behind(obj_pos[0]), 45,
               "Metric 4 demo — broom moves BEHIND O1\n"
               "(B ──→ O1 ──→ GOAL, angle=180° → R4 maximised)")
    snapshot("Metric 4 demo — broom correctly behind O1 (held)", n=25)

    # ── Phase 3: sweep O1 (R1 improves, R2 spikes at entry) ───────────────
    sweep_obj(0, 80, "Sweeping O1 → GOAL\n"
              "R1 rises as distance shrinks, R4 held high by broom tracking")
    snapshot("★ O1 inside goal! R2 bonus fires (+10 per step)", n=30)

    # ── Phases 4-6: sweep remaining objects ───────────────────────────────
    for i in [1, 2, 3]:
        # Metric 3 demo: broom pulled toward nearest unswept object
        move_broom(_behind(obj_pos[i]), 45,
                   f"Metric 3 — broom moves toward nearest unswept O{i+1}\n"
                   f"(R3 improves as broom closes in; R4 activates on arrival)")
        snapshot(f"Broom behind O{i+1}, ready to sweep", n=20)

        sweep_obj(i, 80,
                  f"Sweeping O{i+1} → GOAL\n"
                  f"All 4 metrics active simultaneously")
        snapshot(f"★ O{i+1} swept! R2 = +{(i+1)*10} per step now", n=30)

    # ── Phase 7: all swept ────────────────────────────────────────────────
    snapshot("All objects swept — maximum reward state", n=45)

    return frames


# ── Matplotlib renderer ───────────────────────────────────────────────────
def _render_frame(fig, ax, broom_xy, obj_positions, swept_mask, label):
    ax.clear()
    ax.set_xlim(-0.70, 0.70); ax.set_ylim(-0.70, 0.70)
    ax.set_aspect('equal')
    ax.set_facecolor('#0e0e1c')
    ax.tick_params(colors='#555', labelsize=7)
    for sp in ax.spines.values(): sp.set_edgecolor('#333')
    ax.set_xlabel('X (m)', color='#666', fontsize=8)
    ax.set_ylabel('Y (m)', color='#666', fontsize=8)

    # Goal region
    ax.add_patch(plt.Circle(_GC, _GR, color='#00cc44', alpha=0.18, zorder=1))
    ax.add_patch(plt.Circle(_GC, _GR, fill=False, edgecolor='#00ff66',
                            linewidth=2.0, zorder=2))
    ax.text(_GC[0], _GC[1], 'GOAL', color='#00ff66', ha='center',
            va='center', fontsize=8, fontweight='bold', zorder=3)

    # Reward
    total, rb = compute_reward(
        broom_xy, obj_positions, swept_mask, _GC, _GR
    )
    nearest_idx = rb["nearest_idx"]

    # Triangle: GOAL ─ nearest_obj ─ BROOM (dashed, only for unswept objects)
    if nearest_idx is not None:
        O = np.array(obj_positions[nearest_idx], float)
        B = np.array(broom_xy, float)
        # Goal → Object line
        ax.annotate("", xy=O, xytext=_GC,
                    arrowprops=dict(arrowstyle="->", color="#00ff6655",
                                    lw=1.2, linestyle="dashed"), zorder=3)
        # Object → Broom line
        ax.annotate("", xy=B, xytext=O,
                    arrowprops=dict(arrowstyle="->", color="#ffaa0055",
                                    lw=1.2, linestyle="dashed"), zorder=3)
        # Angle annotation at object vertex
        if rb["angle_deg"] is not None:
            ang_col = '#44ff88' if rb["angle_deg"] > 120 else (
                      '#ffaa44' if rb["angle_deg"] > 60 else '#ff4444')
            ax.text(O[0] + 0.07, O[1] + 0.07,
                    f'{rb["angle_deg"]:.0f}°',
                    color=ang_col, fontsize=8, fontweight='bold', zorder=11)

    # Objects
    for i, (pos, sw, col) in enumerate(zip(obj_positions, swept_mask, _OBJ_COLORS)):
        pos  = np.array(pos, float)
        fill = '#44ff88' if sw else col
        ax.add_patch(plt.Circle(pos, 0.048, color=fill, zorder=5))
        ax.text(pos[0], pos[1], f'O{i+1}', color='#111', ha='center',
                va='center', fontsize=7, fontweight='bold', zorder=6)

    # Broom
    B = np.array(broom_xy, float)
    ax.plot(*B, 's', color='#ffdd00', markersize=15, zorder=7,
            markeredgecolor='#997700', markeredgewidth=1.5)
    ax.text(B[0], B[1] + 0.08, 'BROOM', color='#ffdd00',
            ha='center', fontsize=7, zorder=8)

    # Phase label (title)
    ax.set_title(label, color='white', fontsize=8.5, pad=7,
                 loc='center', linespacing=1.4)

    # ── Reward breakdown panel (right side) ───────────────────────────────
    metrics = [
        ("R1  obj→goal dist", rb['r1_obj_dist'],    '#aaaaff', "closer all objects → less negative"),
        ("R2  swept bonus",   rb['r2_swept_bonus'],  '#44ff88', "+10 per object in goal (per step)"),
        ("R3  broom→nearest", rb['r3_broom_prox'],   '#ffaa44', "closer broom to nearest → less negative"),
        ("R4  angle @ obj",   rb['r4_angle'],         '#ff88ff', "+max when broom behind obj (180°)"),
    ]
    y0  = 0.68
    dy  = 0.115
    x0  = -0.68
    ax.text(x0, y0 + 0.05, "── Reward Breakdown ──",
            color='#aaaaaa', fontsize=7, family='monospace', zorder=10)

    for j, (name, val, col, tip) in enumerate(metrics):
        y = y0 - j * dy
        # Value
        ax.text(x0, y, f"{name}: {val:+.3f}",
                color=col, fontsize=7.5, family='monospace',
                va='center', zorder=10)
        # Tooltip in grey below
        ax.text(x0 + 0.02, y - 0.046, tip,
                color='#666677', fontsize=6, va='center', zorder=10)

    # Divider
    ax.axhline(y0 - 4 * dy, xmin=0.01, xmax=0.52,
               color='#444', linewidth=0.8, zorder=9)

    # Total
    tot_col = '#00ff88' if total >= 0 else '#ff5555'
    ax.text(x0, y0 - 4.3 * dy,
            f"{'TOTAL':>22} {total:+.3f}",
            color=tot_col, fontsize=9, family='monospace',
            va='center', fontweight='bold', zorder=10)
    ax.text(x0, y0 - 5.2 * dy,
            f"Swept: {rb['n_swept']} / 4",
            color='#888899', fontsize=7.5, va='center', zorder=10)


def _fig_to_rgb(fig):
    buf = io.BytesIO()
    # Removed bbox_inches='tight' to guarantee identical frame dimensions
    fig.savefig(buf, format='png', facecolor=fig.get_facecolor(), dpi=100)
    buf.seek(0)
    arr = np.frombuffer(buf.getvalue(), dtype=np.uint8)
    return cv2.cvtColor(cv2.imdecode(arr, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)


# ── Run ───────────────────────────────────────────────────────────────────
print("Building scripted demo frames...")
demo_sequence = _build_demo_sequence()

fig, ax = plt.subplots(figsize=(8, 7), dpi=100)
fig.patch.set_facecolor('#0e0e1c')
plt.tight_layout(pad=1.8)

demo_video = []
for k, (broom, objs, swept, lbl) in enumerate(demo_sequence):
    _render_frame(fig, ax, broom, objs, swept, lbl)
    demo_video.append(_fig_to_rgb(fig))
    if k % 100 == 0:
        print(f"  {k}/{len(demo_sequence)} frames...")

plt.close(fig)
print(f"Done — {len(demo_video)} frames ({len(demo_video)/30:.1f} s at 30 fps)")
media.show_video(demo_video, fps=30)